<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-0/05-limpieza-de-datos/solucion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Solución 0.5 · Limpieza de datos: vacíos, duplicados, textos y fechas

**Serie:** Del Excel al Machine Learning · **Temporada 0** · 🎬 Video próximamente

Estas son las respuestas del ejercicio. Si la tuya es distinta pero da el mismo resultado, ¡también está bien! En programación suele haber más de una forma de llegar a la respuesta.

### Prepara los datos

Ejecuta estas dos celdas. Abren el archivo sucio, arreglan los nombres de columna y quitan los duplicados.

Vas a limpiar la tabla paso a paso, así que **ejecuta los ejercicios en orden**. Si Colab se desconecta o algo sale mal, vuelve a ejecutar todo desde arriba.

> ⚠️ Si ves `Can only use .str accessor`, esa columna ya es número porque la celda ya se ejecutó. No está mal tu código: vuelve a ejecutar desde la celda que abre el archivo.

In [1]:
import pandas as pd

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

In [2]:
ventas = pd.read_csv(DATOS + "ventas_crudo.csv", sep=";")

ventas.columns = ventas.columns.str.lower().str.replace(" ", "_")
ventas = ventas.rename(
    columns={
        "nro_venta": "id_venta",
        "cliente": "id_cliente",
        "producto": "id_producto",
        "vendedor": "id_vendedor",
        "método_de_pago": "metodo_pago",
    }
)
ventas = ventas.drop_duplicates()

ventas.shape

(50000, 11)

### 1. Las ciudades

Muestra cuántas veces aparece cada valor de `ciudad`. Luego quita los espacios sobrantes, deja la primera letra en mayúscula y vuelve a contar: deben quedar **6 ciudades**.

In [3]:
print(ventas["ciudad"].value_counts())

ventas["ciudad"] = ventas["ciudad"].str.strip().str.capitalize()

ventas["ciudad"].value_counts()

ciudad
Lima         17552
Arequipa      7180
Trujillo      6375
Cusco         5253
Chiclayo      4644
Piura         4103
lima           655
Lima           624
LIMA           592
Arequipa       276
arequipa       265
AREQUIPA       254
TRUJILLO       247
Trujillo       212
trujillo       204
Cusco          196
Chiclayo       195
cusco          191
chiclayo       188
CUSCO          183
CHICLAYO       163
Piura          157
PIURA          152
piura          139
Name: count, dtype: int64


ciudad
Lima        19423
Arequipa     7975
Trujillo     7038
Cusco        5823
Chiclayo     5190
Piura        4551
Name: count, dtype: int64

### 2. Los métodos de pago

Unifica `metodo_pago` como en la clase: quita los espacios, deja la primera letra en mayúscula, cambia `Yape` y `Plin` por `Billetera digital`, y llena los vacíos con `Sin dato`. Deben quedar **5 valores**.

In [4]:
ventas["metodo_pago"] = ventas["metodo_pago"].str.strip().str.capitalize()
ventas["metodo_pago"] = ventas["metodo_pago"].replace(
    {"Yape": "Billetera digital", "Plin": "Billetera digital"}
)
ventas["metodo_pago"] = ventas["metodo_pago"].fillna("Sin dato")

ventas["metodo_pago"].value_counts()

metodo_pago
Efectivo             16335
Billetera digital    12573
Transferencia        11158
Crédito               8402
Sin dato              1532
Name: count, dtype: int64

### 3. Tu función para los montos

Crea la función `limpiar_monto`, que reciba una columna y la entregue convertida en número. Úsala con `precio_unitario` y con `total`, y muestra la suma de `total`.

> 💡 Pista: dentro de la función va lo mismo que hicimos en la clase, pero con `columna.str.replace(...)`.

In [5]:
def limpiar_monto(columna):
    return columna.str.replace("S/ ", "").str.replace(",", "").astype(float)


ventas["precio_unitario"] = limpiar_monto(ventas["precio_unitario"])
ventas["total"] = limpiar_monto(ventas["total"])

print(ventas["total"].sum())

13582286.61


### 4. Los descuentos

La columna `descuento` trae textos como `5%`. Conviértela en un número de `0` a `1`, donde `5%` sea `0.05`, y muestra cuántas veces aparece cada valor.

> 💡 Pista: quita el `%` con `.str.replace()`, convierte en número y divide toda la columna entre `100`.

In [6]:
ventas["descuento"] = ventas["descuento"].str.replace("%", "").astype(float) / 100

ventas["descuento"].value_counts()

descuento
0.00    47102
0.05     2775
0.10      123
Name: count, dtype: int64

### 5. Las fechas

Convierte `fecha` en una fecha de verdad, con los dos formatos que vimos en la clase. Comprueba que no quede ninguna vacía. Luego crea las columnas `anio` y `mes`.

> 💡 Pista: una columna nueva se crea igual que una variable: `ventas["mes"] = ...`.

In [7]:
texto = ventas["fecha"]

ventas["fecha"] = pd.to_datetime(texto, format="%d/%m/%Y", errors="coerce")
ventas["fecha"] = ventas["fecha"].fillna(pd.to_datetime(texto, format="%Y-%m-%d", errors="coerce"))

print(ventas["fecha"].isna().sum())

ventas["anio"] = ventas["fecha"].dt.year
ventas["mes"] = ventas["fecha"].dt.month

ventas[["fecha", "anio", "mes"]].head()

0


,fecha,anio,mes
0,2024-01-02,2024,1
1,2024-01-02,2024,1
2,2024-01-02,2024,1
3,2024-01-02,2024,1
4,2024-01-02,2024,1


### 6. Las cantidades

La columna `cantidad` tiene vacíos y errores de tipeo. Por suerte, se puede recalcular con esta fórmula:

`total / (precio_unitario * (1 - descuento))`

> 💡 Pista: cada nombre es una columna, como `ventas["total"]`.

Guarda ese cálculo, redondeado con `.round()`, en una variable llamada `cantidad`. **Antes de cambiar nada**, cuenta cuántas filas son distintas de la columna actual, con `(ventas["cantidad"] != cantidad).sum()`: deben ser `514`.

Después, guárdalo en la columna `cantidad`, convertido en entero con `.astype(int)`, y muestra el valor **más alto** y cuántos vacíos quedan: debe ser `41` y `0`.

In [8]:
cantidad = ventas["total"] / (ventas["precio_unitario"] * (1 - ventas["descuento"]))
cantidad = cantidad.round()

print((ventas["cantidad"] != cantidad).sum())

ventas["cantidad"] = cantidad.astype(int)

print(ventas["cantidad"].max())
print(ventas["cantidad"].isna().sum())

514
41
0


### 7. ¿Cuadra todo?

Abre el archivo limpio oficial, `ventas.csv`, en una variable llamada `oficial`. Comprueba que las dos tablas cuadren: muestra la cantidad de filas de cada una con `len()`, la suma de `total` de cada una, redondeada a 2 decimales, y la suma de `cantidad`.

In [9]:
oficial = pd.read_csv(DATOS + "ventas.csv")

print(len(ventas), len(oficial))
print(round(ventas["total"].sum(), 2), round(oficial["total"].sum(), 2))
print(ventas["cantidad"].sum(), oficial["cantidad"].sum())

50000 50000
13582286.61 13582286.61
192872 192872


Las dos tablas tienen 50,000 filas y suman lo mismo. En los valores, la única diferencia es `Sin dato` en `metodo_pago`: esa información se perdió en el archivo sucio, y no se puede recuperar. Además, nuestra tabla tiene las columnas `anio` y `mes`, y `fecha` ya es una fecha de verdad.

## 🎉 ¡Bien hecho!

Nos vemos en el **episodio 0.6: pandas II, tablas dinámicas, BUSCARV y estadística básica**.

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0